In [ ]:
# 要解决循环数组的最长非空连续子数组和问题（注：根据您的示例 [5,-1,5]，您实际需要的是 “连续子数组和”，而非 “子序列”—— 子序列不要求连续，子数组需连续），需结合普通数组的最长子数组和算法（Kadane 算法），并针对 “循环” 特性补充特殊处理。
# 一、核心思路：两种可能的最长子数组
# 循环数组的最长连续子数组有两种形态，需分别计算后取最大值：

# 不跨首尾：子数组在原数组内部（如 [1,2,3] 的最长子数组 [1,2,3]），与普通数组的最长子数组和一致，直接用 Kadane 算法求解。
# 跨首尾：子数组包含数组末尾和开头元素（如 [5,-1,5] 的最长子数组 [5（末尾）,5（开头）]）。此时，跨首尾的子数组和 = 数组总和 - 普通数组的最短连续子数组和（本质是 “去掉中间的最短子数组，剩余部分即跨首尾的最长子数组”）。
# 二、关键注意点：排除 “空数组” 陷阱
# 当数组全为负数时，“数组总和 - 最短子数组和” 会等于 0（因为最短子数组是整个数组，总和 = 最短子数组和），但子数组要求 “非空”，因此这种情况需直接取 “普通数组的最长子数组和”（即数组中最大的单个负数）。
# 三、具体实现步骤
# 步骤 1：处理边界情况
# 若数组长度为 1，直接返回该唯一元素（循环无意义，最长子数组就是自身）。
# 步骤 2：计算数组总和（total）
# 用于后续计算 “跨首尾子数组和”。
# 步骤 3：用 Kadane 算法计算普通数组的 “最长子数组和”（max_sub）
# Kadane 算法核心：遍历数组，维护 “当前子数组和”，若当前元素比 “当前子数组和 + 当前元素” 更大，则重置当前子数组为 “仅当前元素”，同时更新全局最大值。
# 步骤 4：用类似 Kadane 的算法计算普通数组的 “最短子数组和”（min_sub）
# 逻辑与 Kadane 算法相反：维护 “当前子数组和”，若当前元素比 “当前子数组和 + 当前元素” 更小，则重置当前子数组为 “仅当前元素”，同时更新全局最小值。
# 步骤 5：计算 “跨首尾子数组和”（cross_max）
# cross_max = total - min_sub
# 步骤 6：确定最终结果
# 若cross_max == 0（说明数组全为负数，cross_max 对应空数组，无效），结果为max_sub；
# 否则，结果为max(max_sub, cross_max)。
# 四、示例验证
# 以您的示例[5,-1,5]为例：

# 边界情况：长度 = 3≠1，跳过；
# 总和total = 5 + (-1) + 5 = 9；
# 计算max_sub：
# 初始current_max=5，max_sub=5；
# 遍历到 - 1：current_max = max(-1, 5-1=4) = 4，max_sub=5；
# 遍历到 5：current_max = max(5, 4+5=9) = 9，max_sub=9；
# 计算min_sub：
# 初始current_min=5，min_sub=5；
# 遍历到 - 1：current_min = min(-1, 5-1=4) = -1，min_sub=-1；
# 遍历到 5：current_min = min(5, -1+5=4) = 4，min_sub=-1；
# cross_max = 9 - (-1) = 10；
# cross_max≠0，最终结果max(9,10)=10（即跨首尾的 [5,5]，和为 10），与您的预期一致。
# 五、代码实现（Python）
# python
def max_circular_subarray_sum(arr):
    n = len(arr)
    # 边界情况：数组长度为1
    if n == 1:
        return arr[0]
    
    # 步骤1：计算数组总和
    total = sum(arr)
    
    # 步骤2：Kadane算法求普通数组的最长子数组和（max_sub）
    current_max = max_sub = arr[0]
    for num in arr[1:]:
        current_max = max(num, current_max + num)
        max_sub = max(max_sub, current_max)
    
    # 步骤3：类似Kadane求普通数组的最短子数组和（min_sub）
    current_min = min_sub = arr[0]
    for num in arr[1:]:
        current_min = min(num, current_min + num)
        min_sub = min(min_sub, current_min)
    
    # 步骤4：计算跨首尾的可能最大和，排除空数组情况
    cross_max = total - min_sub
    # 若cross_max为0，说明数组全负，取max_sub（最大的单个负数）
    result = max_sub if cross_max == 0 else max(max_sub, cross_max)
    
    return result

# 测试示例
print(max_circular_subarray_sum([5, -1, 5]))  # 输出10
print(max_circular_subarray_sum([-5, -3, -2]))  # 输出-2（全负，取最大单个负数）
print(max_circular_subarray_sum([2, -1, 2]))  # 输出4（跨首尾的[2,2]）
print(max_circular_subarray_sum([1]))  # 输出1

In [8]:
# 一个数组长度为n，要求求这个循环数组的最长非空连续子序列，注意，这个数组的咨询可以循环来搞，例如[5，-1,5] 最长子序列就是5，连续数组应该使用前缀和等trick
test_arr=[5,-1,5]
def solution(arr):
    n=len(arr)
    # for i in arr:
        # arr.append(i)
    tmp=arr[:]
    arr=arr+tmp
    N=2*n
    dp=[[0 for _ in range(N+1)]for _ in range(N+1)]
    prefix_sum=0
    pre_sum=[0 for _ in range(N+1)]
    for i in range(0,N):
        prefix_sum+=arr[i]
        pre_sum[i]=prefix_sum
    pre_sum[-1]=0
    ans=max(arr)
    for i in range(0,N):
        cur=0
        for j in range(N-1,i-1,-1):
            if j-i >=n:
                continue
            else:
                print(i,j,pre_sum[j]-pre_sum[i-1])
                # pre_sum[i]
                ans=max(ans,pre_sum[j]-pre_sum[i-1])
    print("答案是",ans)
                
solution(test_arr)
solution([3,4,-100,6,7,-1,-100,1,4])

0 2 9
0 1 4
0 0 5
1 3 9
1 2 4
1 1 -1
2 4 9
2 3 10
2 2 5
3 5 9
3 4 4
3 3 5
4 5 4
4 4 -1
5 5 5
答案是 10
0 8 -176
0 7 -180
0 6 -181
0 5 -81
0 4 -80
0 3 -87
0 2 -93
0 1 7
0 0 3
1 9 -176
1 8 -179
1 7 -183
1 6 -184
1 5 -84
1 4 -83
1 3 -90
1 2 -96
1 1 4
2 10 -176
2 9 -180
2 8 -183
2 7 -187
2 6 -188
2 5 -88
2 4 -87
2 3 -94
2 2 -100
3 11 -176
3 10 -76
3 9 -80
3 8 -83
3 7 -87
3 6 -88
3 5 12
3 4 13
3 3 6
4 12 -176
4 11 -182
4 10 -82
4 9 -86
4 8 -89
4 7 -93
4 6 -94
4 5 6
4 4 7
5 13 -176
5 12 -183
5 11 -189
5 10 -89
5 9 -93
5 8 -96
5 7 -100
5 6 -101
5 5 -1
6 14 -176
6 13 -175
6 12 -182
6 11 -188
6 10 -88
6 9 -92
6 8 -95
6 7 -99
6 6 -100
7 15 -176
7 14 -76
7 13 -75
7 12 -82
7 11 -88
7 10 12
7 9 8
7 8 5
7 7 1
8 16 -176
8 15 -177
8 14 -77
8 13 -76
8 12 -83
8 11 -89
8 10 11
8 9 7
8 8 4
9 17 -176
9 16 -180
9 15 -181
9 14 -81
9 13 -80
9 12 -87
9 11 -93
9 10 7
9 9 3
10 17 -179
10 16 -183
10 15 -184
10 14 -84
10 13 -83
10 12 -90
10 11 -96
10 10 4
11 17 -183
11 16 -187
11 15 -188
11 14 -88
11 13 -87
11 12 -94